# <font color="#418FDE" size="6.5" uppercase>**SVR Kernel Ridge**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply SVR variants to regression problems with suitable epsilon, C, and kernel settings. 
- Use Kernel Ridge regression and compare it with SVR on accuracy and computational behavior. 
- Prepare data for kernel methods using scaling and precomputed-kernel conventions. 


## **1. SVR Variants**

### **1.1. Epsilon in SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_01.jpg?v=1783844527" width="250">



>* Epsilon defines an error tolerance zone.
>* Larger epsilon ignores small noise variations.

>* Epsilon balances noise sensitivity and robustness.
>* Match epsilon to scale and stakes.

>* Epsilon interacts with C and kernel.
>* Choose tolerance based on task error.



### **1.2. Linear SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_02.jpg?v=1783844525" width="250">



>* Fits roughly linear patterns with noise tolerance
>* Works well for high-dimensional structured data

>* C controls fit versus noise sensitivity.
>* Epsilon sets ignored error size.

>* Efficient for large, high-dimensional linear problems.
>* Baseline model; weaker on nonlinear patterns.



### **1.3. Nu SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_03.jpg?v=1783844530" width="250">



>* Nu SVR controls errors without choosing epsilon.
>* Tune nu, C, and kernel for balance.

>* Nu balances sparsity and responsiveness.
>* Its effect depends on C and kernel.

>* Scale features and choose an appropriate kernel.
>* Tune nu and C; inspect fit quality.



## **2. Kernel Ridge Comparison**

### **2.1. Kernel Ridge Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_01.jpg?v=1783844532" width="250">



>* Combines ridge regularization with kernel flexibility.
>* Models smooth nonlinear patterns while limiting overfitting.

>* Predictions come from similarity to training cases.
>* Ridge regularization stabilizes complex nonlinear relationships.

>* Simple training, strong smooth nonlinear predictions.
>* Large datasets increase cost and prediction dependence.



### **2.2. SVR Versus Kernel Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_02.jpg?v=1783844536" width="250">



>* SVR ignores small errors, uses support vectors.
>* Kernel Ridge fits all points smoothly.

>* Accuracy depends on data and noise.
>* SVR handles noise; KRR fits smooth trends.

>* Kernel Ridge trains simply, predicts using all points.
>* SVR trains harder, but predicts sparsely faster.



### **2.3. SVR KRR Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_03.jpg?v=1783844537" width="250">



>* KRR uses all points for smoothness.
>* SVR ignores small errors, using support vectors.

>* KRR trains simply but predicts slowly large-scale.
>* SVR trains harder but can predict faster.

>* KRR is smoother and easier to tune.
>* Compare speed, memory, tuning, and noise handling.



## **3. Kernel Method Limits**

### **3.1. Scaling for Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_01.jpg?v=1783844522" width="250">



>* Scaling prevents large-range features dominating similarity.
>* It makes kernels reflect real patterns.

>* Scaling preserves meaningful kernel distance relationships.
>* It also makes tuning more reliable.

>* Fit scaling on training, reuse everywhere.
>* Scaling defines stable kernel similarity.



In [ ]:
#@title Python Code - Scaling for Kernels

# Scaling changes kernel distance comparisons strongly.
# This example uses only NumPy math.
# Mixed units can distort similarity badly.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two features with mixed units.
size_sqft = np.linspace(600, 3000, 40)

distance_miles = rng.uniform(0.5, 20.0, 40)
noise = rng.normal(0.0, 15.0, 40)

# Build a simple target for context.
y = 120 + 0.08 * size_sqft

y = y - 6.0 * distance_miles + noise
X = np.column_stack((size_sqft, distance_miles))

# Check the feature matrix shape.
assert X.shape == (40, 2), "Unexpected feature shape"

# Standardize using training-style statistics.
means = X.mean(axis=0)
stds = X.std(axis=0)
stds = np.where(stds == 0, 1.0, stds)
X_scaled = (X - means) / stds

# Define a small RBF kernel helper.
def rbf_similarity(a, b, gamma):
    diff = a - b

    return np.exp(-gamma * np.dot(diff, diff))

# Compare one home against two others.
anchor = X[0]
near_size_far_distance = X[10]
far_size_near_distance = X[30]

# Compute similarities before scaling.
gamma = 0.5
raw_sim_a = rbf_similarity(anchor, near_size_far_distance, gamma)
raw_sim_b = rbf_similarity(anchor, far_size_near_distance, gamma)

# Compute similarities after scaling.
anchor_s = X_scaled[0]
point_a_s = X_scaled[10]
point_b_s = X_scaled[30]
scaled_sim_a = rbf_similarity(anchor_s, point_a_s, gamma)
scaled_sim_b = rbf_similarity(anchor_s, point_b_s, gamma)

# Print a short teaching summary.
print("Feature scales:", np.round(stds, 2))
print("Raw similarity A:", round(raw_sim_a, 6))
print("Raw similarity B:", round(raw_sim_b, 6))
print("Scaled similarity A:", round(scaled_sim_a, 6))
print("Scaled similarity B:", round(scaled_sim_b, 6))
print("Scaling makes both features matter more fairly.")

# Plot raw and scaled feature spaces.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Show the unscaled geometry.
axes[0].scatter(X[:, 0], X[:, 1], color="steelblue")
axes[0].scatter(anchor[0], anchor[1], color="red", s=70)
axes[0].set_title("Before scaling")
axes[0].set_xlabel("Size in square feet")

axes[0].set_ylabel("Distance in miles")

# Show the scaled geometry.
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], color="seagreen")
axes[1].scatter(anchor_s[0], anchor_s[1], color="red", s=70)
axes[1].set_title("After scaling")
axes[1].set_xlabel("Scaled size")

axes[1].set_ylabel("Scaled distance")
plt.tight_layout()
plt.show()



### **3.2. Kernel Scaling Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_02.jpg?v=1783844520" width="250">



>* Large-scale features can distort kernel similarity.
>* Scaling balances features, especially for distances.

>* Scaling reshapes feature space for fairer similarity.
>* It stabilizes distances and clarifies kernel settings.

>* Reuse training-set scaling for all future data.
>* Consistent scaling keeps kernel comparisons reliable.



In [ ]:
#@title Python Code - Kernel Scaling Basics

# Kernel scaling changes distance based similarity.
# This example compares raw and scaled distances.
# It also shows a precomputed kernel matrix.

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

# Set a deterministic random seed.
np.random.seed(7)

# Build tiny data with mixed units.
X_train = np.array([
    [1800.0, 2.0],
    [2200.0, 8.0],
    [1600.0, 3.0],
])

X_new = np.array([
    [2000.0, 4.0],
    [2100.0, 7.0],
])

# Measure raw Euclidean distances.
raw_dist = np.linalg.norm(
    X_train[0] - X_train[1]
)

# Fit scaling only on training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_new_scaled = scaler.transform(X_new)

# Measure scaled Euclidean distances.
scaled_dist = np.linalg.norm(
    X_train_scaled[0] - X_train_scaled[1]
)

# Build precomputed RBF kernel matrices.
K_train = rbf_kernel(
    X_train_scaled, X_train_scaled, gamma=0.5
)
K_new = rbf_kernel(
    X_new_scaled, X_train_scaled, gamma=0.5
)

# Validate expected kernel shapes.
assert K_train.shape == (3, 3)
assert K_new.shape == (2, 3)

# Print short teaching results.
print("Raw distance:", round(raw_dist, 2))
print("Scaled distance:", round(scaled_dist, 2))
print("Train means:", np.round(scaler.mean_, 2))
print("Kernel train shape:", K_train.shape)
print("Kernel new shape:", K_new.shape)
print("First new similarities:", np.round(K_new[0], 3))



### **3.3. Precomputed Kernel Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_03.jpg?v=1783844524" width="250">



>* Precomputed kernels supply similarities as a matrix.
>* Useful when custom similarity better fits data.

>* Training and prediction kernels need correct structure.
>* Keep example order and similarity rules consistent.

>* Precomputed kernels rely on trustworthy similarities.
>* Consistency and validation prevent misleading predictions.



# <font color="#418FDE" size="6.5" uppercase>**SVR Kernel Ridge**</font>


In this lecture, you learned to:
- Apply SVR variants to regression problems with suitable epsilon, C, and kernel settings. 
- Use Kernel Ridge regression and compare it with SVR on accuracy and computational behavior. 
- Prepare data for kernel methods using scaling and precomputed-kernel conventions. 

In the next Module (Module 13), we will go over 'None'